# Belebele baseline

Zero-shot English/Ukrainian reading comprehension. English instructions in both languages. Select the highest-probability single-token label among A-D; this is constrained choice scoring, not free-text generation.

Run in Slurm. `MODEL_NAME` selects `gemma` or `lapa`. `ITEM_LIMIT=2` checks the runtime; `ITEM_LIMIT=0` evaluates all records. Outputs are saved outside Git.

`SHARD_INDEX` and `SHARD_COUNT` divide computation across GPUs. Combined shards contain the same evaluation records.

Saved outputs below: Gemma, shard 0 of 2, full run `2131431-limit0`. Each shard contains half the records; the results notebooks combine both shards and both models. Rerunning requires the Slurm environment.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json
import os
import platform
import time

import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from transformers import AutoTokenizer, Gemma3ForConditionalGeneration

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
project = Path("/e/project1/e-dev-2026d09-075")
model_name = os.environ.get("MODEL_NAME", "gemma")
limit = int(os.environ.get("ITEM_LIMIT", "2"))
assert limit >= 0
shard = int(os.environ.get("SHARD_INDEX", "0"))
shards = int(os.environ.get("SHARD_COUNT", "1"))
assert 0 <= shard < shards
models = {
    "gemma": project
    / "fairforget-steering-transfer/models/gemma-3-12b-it/96b6f1eccf38110c56df3a15bffe176da04bfd80",
    "lapa": project
    / "fairforget/models/lapa-v0.1.2-instruct/2969fd997f07b33727c6f10795d78ef23cc5c037",
}
model_path = models[model_name]
run_id = os.environ.get(
    "RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
)
run_dir = root / "results" / "baselines" / run_id / "belebele" / model_name
run_dir = run_dir / f"shard-{shard}"
run_dir.mkdir(parents=True, exist_ok=True)
torch.manual_seed(42)
torch.set_num_threads(8)
print("Model:", model_name, model_path.name)
print("Items per language/source:", limit if limit else "all")
print("Results:", run_dir)

Model: gemma 96b6f1eccf38110c56df3a15bffe176da04bfd80
Items per language/source: all
Results: /e/project1/e-dev-2026d09-075/fairforget-steering-transfer/results/baselines/2131431-limit0/belebele/gemma/shard-0


## Data

Load the pinned English and Ukrainian files, then select this worker's shard. Shards divide computation; they are not train/test splits.

In [2]:
revision = "7899cdfa4e1e0d733fd77c848e2c273cb1d32be2"
frames, input_hashes = {}, {}
keys = ["link", "question_number"]
for language in ["eng_Latn", "ukr_Cyrl"]:
    path = Path(
        hf_hub_download(
            "facebook/belebele",
            f"data/{language}.jsonl",
            repo_type="dataset",
            revision=revision,
        )
    )
    input_hashes[language] = hashlib.sha256(path.read_bytes()).hexdigest()
    frame = pd.read_json(path, lines=True).sort_values(keys).reset_index(drop=True)
    assert len(frame) == 900 and not frame.duplicated(keys).any()
    assert frame.correct_answer_num.isin([1, 2, 3, 4]).all()
    selected = frame.head(limit) if limit else frame
    frames[language] = selected.iloc[shard::shards]
paired = frames["eng_Latn"].merge(
    frames["ukr_Cyrl"], on=keys, suffixes=("_en", "_uk"), validate="one_to_one"
)
assert len(paired) == len(frames["eng_Latn"]) == len(frames["ukr_Cyrl"])
assert paired.correct_answer_num_en.eq(paired.correct_answer_num_uk).all()
print({language: len(frame) for language, frame in frames.items()})

{'eng_Latn': 450, 'ukr_Cyrl': 450}


## Model

Load the selected checkpoint in BF16 on one allocated GPU. Record the revision, environment, and chat template with the predictions.

In [3]:
assert torch.cuda.is_available(), "Run inside a GPU Slurm allocation"
print("Host:", platform.node())
print("GPU:", torch.cuda.get_device_name(0))
print("Visible GPUs:", torch.cuda.device_count())
assert torch.cuda.device_count() == 1, "Each worker must see exactly one GPU"
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = Gemma3ForConditionalGeneration.from_pretrained(
    model_path,
    dtype=torch.bfloat16,
    device_map={"": "cuda:0"},
    attn_implementation="sdpa",
    local_files_only=True,
).eval()
print("Model class:", type(model).__name__)
print("Layers:", model.config.text_config.num_hidden_layers)
print("Hidden size:", model.config.text_config.hidden_size)
print("Device:", model.device, "dtype:", model.dtype)
assert model.device.type == "cuda"
metadata = {
    "task": "belebele",
    "model": model_name,
    "model_path": str(model_path),
    "revision": model_path.name,
    "input_sha256": input_hashes,
    "seed": 42,
    "shard_index": shard,
    "shard_count": shards,
    "item_limit": limit,
    "run_id": run_id,
    "model_class": type(model).__name__,
    "dtype": str(model.dtype),
    "attention": "sdpa",
    "gpu": torch.cuda.get_device_name(0),
    "hostname": platform.node(),
    "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "python": platform.python_version(),
    "versions": {
        name: version(name) for name in ["torch", "transformers", "pandas", "nbclient"]
    },
    "chat_template": tokenizer.chat_template,
}

Host:

 jpbo-057-32.jupiter.internal


GPU: NVIDIA GH200 120GB
Visible GPUs: 1


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Model class: Gemma3ForConditionalGeneration
Layers: 48
Hidden size: 3840
Device: cuda:0 dtype: torch.bfloat16


## Prompt and scoring

Use the same English instruction for both languages. Score the next-token logits for A-D and choose the largest; the softmax below is normalized only over these four labels.

In [4]:
def make_prompt(row):
    options = "\n".join(
        f"{letter}. {row[f'mc_answer{number}']}"
        for number, letter in enumerate("ABCD", start=1)
    )
    return (
        "Read the passage and answer the question. Choose the correct option. "
        "Reply with only A, B, C, or D.\n\n"
        f"Passage:\n{row['flores_passage']}\n\n"
        f"Question: {row['question']}\n{options}"
    )


labels = list("ABCD")
label_tokens = [tokenizer.encode(label, add_special_tokens=False) for label in labels]
assert all(len(tokens) == 1 for tokens in label_tokens), (
    "Check multi-token label scoring"
)
label_ids = [tokens[0] for tokens in label_tokens]
assert len(set(label_ids)) == 4
print(dict(zip(labels, label_ids)))
example = make_prompt(frames["eng_Latn"].iloc[0])
print(
    tokenizer.apply_chat_template(
        [{"role": "user", "content": example}],
        tokenize=False,
        add_generation_prompt=True,
    )
)
metadata.update(
    {
        "scoring": "next_token_argmax_ABCD",
        "label_ids": dict(zip(labels, label_ids)),
        "instruction_language": "English",
        "few_shot_examples": 0,
    }
)
(run_dir / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False)
)

{'A': 236776, 'B': 236799, 'C': 236780, 'D': 236796}
<bos><start_of_turn>user
Read the passage and answer the question. Choose the correct option. Reply with only A, B, C, or D.

Passage:
Make sure your hand is as relaxed as possible while still hitting all the notes correctly - also try not to make much extraneous motion with your fingers. This way, you will tire yourself out as little as possible. Remember there's no need to hit the keys with a lot of force for extra volume like on the piano. On the accordion, to get extra volume, you use the bellows with more pressure or speed.

Question: According to the passage, what would not be considered an accurate tip for successfully playing the accordion?
A. For additional volume, increase the force with which you hit the keys
B. Keep unnecessary movement to a minimum in order to preserve your stamina
C. Be mindful of hitting the notes while maintaining a relaxed hand
D. Increase the speed with which you operate the bellows to achieve extra

2701

## Run

Evaluate each question without sampling or answer-key text in the prompt. Save its choice, answer key, and four choice scores.

In [5]:
output_path = run_dir / "predictions.jsonl"
assert not output_path.exists(), "Choose a new RUN_ID; do not overwrite predictions"
started = time.monotonic()
with output_path.open("x") as output:
    for language, frame in frames.items():
        for count, (_, row) in enumerate(frame.iterrows(), start=1):
            prompt = make_prompt(row)
            inputs = tokenizer.apply_chat_template(
                [{"role": "user", "content": prompt}],
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            ).to(model.device)
            with torch.inference_mode():
                logits = (
                    model(**inputs, use_cache=False, logits_to_keep=1)
                    .logits[0, -1, label_ids]
                    .float()
                )
            assert torch.isfinite(logits).all()
            probabilities = logits.softmax(dim=-1).cpu().tolist()
            prediction = labels[int(logits.argmax())]
            answer = labels[int(row.correct_answer_num) - 1]
            result = {
                "language": language,
                "link": row.link,
                "question_number": int(row.question_number),
                "prediction": prediction,
                "answer": answer,
                "correct": prediction == answer,
                "label_probabilities": dict(zip(labels, probabilities)),
                "input_tokens": int(inputs.input_ids.shape[-1]),
            }
            output.write(json.dumps(result, ensure_ascii=False) + "\n")
            output.flush()
            if count % 100 == 0 or count == len(frame):
                print(
                    language,
                    count,
                    "/",
                    len(frame),
                    "elapsed seconds",
                    round(time.monotonic() - started),
                )

eng_Latn 100 / 450 elapsed seconds 6


eng_Latn 200 / 450 elapsed seconds 12


eng_Latn 300 / 450 elapsed seconds 18


eng_Latn 400 / 450 elapsed seconds 24


eng_Latn 450 / 450 elapsed seconds 27


ukr_Cyrl 100 / 450 elapsed seconds 33


ukr_Cyrl 200 / 450 elapsed seconds 39


ukr_Cyrl 300 / 450 elapsed seconds 45


ukr_Cyrl 400 / 450 elapsed seconds 51


ukr_Cyrl 450 / 450 elapsed seconds 55


## Results

This table covers one worker. Notebook 06 combines all workers; notebook 07 computes passage-level uncertainty.

In [6]:
predictions = pd.read_json(output_path, lines=True)
summary = predictions.groupby("language").correct.agg(["count", "sum", "mean"])
summary.columns = ["items", "correct", "accuracy"]
display(summary)
display(predictions.head(4))
summary.to_csv(run_dir / "accuracy.csv")
assert len(predictions) == sum(len(frame) for frame in frames.values())
(run_dir / "complete.json").write_text(
    json.dumps(
        {"records": len(predictions), "elapsed_seconds": time.monotonic() - started}
    )
)

,items,correct,accuracy
language,,,
eng_Latn,450,419,0.931111
ukr_Cyrl,450,407,0.904444


,language,link,question_number,prediction,answer,correct,label_probabilities,input_tokens
0,eng_Latn,https://en.wikibooks.org/wiki/Accordion/Right_...,1,A,A,True,"{'A': 0.9999958276748651, 'B': 2.5612780518713...",206
1,eng_Latn,https://en.wikibooks.org/wiki/All_About_Conver...,1,B,B,True,"{'A': 9.931192579415438e-08, 'B': 0.9999997615...",204
2,eng_Latn,https://en.wikibooks.org/wiki/American_Revolut...,2,C,C,True,"{'A': 1.27519015791222e-07, 'B': 1.27519015791...",191
3,eng_Latn,https://en.wikibooks.org/wiki/American_Revolut...,2,C,C,True,"{'A': 1.855382265603111e-07, 'B': 5.7149793519...",237


54